In [ ]:
# Importação de pacotes e módulos
import copy
import gc
import importlib
import itertools
import json
import logging
import os
import subprocess
import sys
import time

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

PKGS = {
    "adabelief_pytorch": "adabelief-pytorch==0.2.1",
    "came_pytorch": "came-pytorch",
    "grokfast_pytorch": "grokfast-pytorch",
    "lion_pytorch": "lion-pytorch",
    "pytorch_optimizer": "pytorch-optimizer",
    "sophia_opt": "sophia-opt",
    "ultralytics": "ultralytics",
    "roboflow": "roboflow",
    "onnx": "onnx",
    "onnxruntime": "onnxruntime",
    "onnxsim": "onnxsim",
    "ncnn": "ncnn",
    "torch_pruning": "torch-pruning",
    "thop": "thop",
}

for module, pip_name in PKGS.items():
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "-q"])

import onnxruntime as ort
import torch_pruning as tp

from adabelief_pytorch import AdaBelief
from came_pytorch import CAME
from grokfast_pytorch import GrokFastAdamW
from lion_pytorch import Lion
from pytorch_optimizer import AdEMAMix
from sophia_opt import SophiaG
from thop import profile
from torch.nn.utils import prune
from ultralytics import YOLO
from ultralytics.engine.trainer import BaseTrainer
from ultralytics.nn.modules.conv import Conv

logging.getLogger("ultralytics").setLevel(logging.WARNING)

In [ ]:
# Define globalmente a ativação dos blocos Conv do YOLO.
def force_yolo_activation(activation_name):
    if activation_name == "SiLU":
        Conv.default_act = nn.SiLU()
    elif activation_name == "Hardswish":
        Conv.default_act = nn.Hardswish(inplace=True)
    elif activation_name == "LeakyReLU":
        Conv.default_act = nn.LeakyReLU(negative_slope=0.1, inplace=True)
    elif activation_name == "GELU":
        Conv.default_act = nn.GELU()
    elif activation_name == "Mish":
        Conv.default_act = nn.Mish(inplace=True)
    elif activation_name == "ELU":
        Conv.default_act = nn.ELU(inplace=True)
    elif activation_name == "PReLU":
        Conv.default_act = nn.PReLU()
    else:
        raise ValueError(f"Ativação {activation_name} não suportada!")


# Aplica monkey patch para suportar otimizadores alternativos.
if not hasattr(BaseTrainer, "_already_patched"):
    _original_build_optimizer = BaseTrainer.build_optimizer

    def custom_build_optimizer(
        self, model, name="auto", lr=0.01, momentum=0.9, decay=1e-5, iterations=1e5
    ):
        opt_name = getattr(self, "force_custom_opt", name)

        g = [], [], []
        bn = tuple(v for k, v in nn.__dict__.items() if "Norm" in k)
        for v in model.modules():
            if hasattr(v, "bias") and isinstance(v.bias, nn.Parameter):
                g[2].append(v.bias)
            if isinstance(v, bn):
                g[1].append(v.weight)
            elif hasattr(v, "weight") and isinstance(v.weight, nn.Parameter):
                g[0].append(v.weight)

        if opt_name == "AdaBelief":
            self.optimizer = AdaBelief(
                g[0],
                lr=lr,
                eps=1e-16,
                betas=(0.9, 0.999),
                weight_decouple=True,
                rectify=True,
                print_change_log=False,
            )
        elif opt_name == "Lion":
            self.optimizer = Lion(g[0], lr=lr, weight_decay=1e-4)
        elif opt_name == "SophiaG":
            self.optimizer = SophiaG(
                g[0], lr=lr, betas=(0.965, 0.99), rho=0.01, weight_decay=1e-4
            )
        elif opt_name == "CAME":
            self.optimizer = CAME(
                g[0], lr=lr, weight_decay=1e-4, betas=(0.9, 0.999, 0.9999)
            )
        elif opt_name == "Grokfast":
            self.optimizer = GrokFastAdamW(g[0], lr=lr, weight_decay=1e-4)
        elif opt_name == "AdEMAMix":
            self.optimizer = AdEMAMix(
                g[0], lr=lr, betas=(0.9, 0.999, 0.9999), weight_decay=1e-4
            )
        else:
            return _original_build_optimizer(
                self, model, opt_name, lr, momentum, decay, iterations
            )

        self.optimizer.add_param_group({"params": g[1], "weight_decay": 0.0})
        self.optimizer.add_param_group({"params": g[2], "weight_decay": 0.0})
        return self.optimizer

    BaseTrainer.build_optimizer = custom_build_optimizer
    BaseTrainer._already_patched = True


# Parâmetros do grid search.
optimizers_to_test = [
    "AdamW",
    "NAdam",
    "RMSProp",
    "SGD",
    "AdaBelief",
    "Lion",
    "AdEMAMix",
    "SophiaG",
    "CAME",
    "Grokfast",
]

activations_to_test = ["SiLU", "LeakyReLU", "Hardswish", "GELU", "Mish", "ELU", "PReLU"]

dataset_yaml = "/workspace/projeto/main.yaml"
pasta_base = "/workspace/projeto/MBA_Polyp_Tests"

optimizer_hyperparams = {
    "SGD": {"lr0": 0.01},
    "AdamW": {"lr0": 0.01},
    "NAdam": {"lr0": 0.01},
    "RMSProp": {"lr0": 0.01},
    "AdaBelief": {"lr0": 0.001},
    "Lion": {"lr0": 0.0001},
    "AdEMAMix": {"lr0": 0.001},
    "SophiaG": {"lr0": 0.0001},
    "CAME": {"lr0": 0.001},
    "Grokfast": {"lr0": 0.001},
}


# Resume as métricas principais de segmentação.
def extract_seg_metrics(metrics):
    return {
        "mAP50_box": round(float(metrics.box.map50), 4),
        "mAP50_mask": round(float(metrics.seg.map50), 4),
        "mAP50-95_box": round(float(metrics.box.map), 4),
        "mAP50-95_mask": round(float(metrics.seg.map), 4),
        "precision_mask": round(float(metrics.seg.mp), 4),
        "recall_mask": round(float(metrics.seg.mr), 4),
    }


# Limpa cache e libera VRAM após cada execução.
def cleanup_memory():
    gc.collect()
    torch.cuda.empty_cache()


# Instancia o modelo de acordo com a ativação escolhida.
def build_model_for_activation(act_name):
    force_yolo_activation(act_name)
    if act_name == "SiLU":
        return YOLO("yolov8n-seg.pt")

    model = YOLO("yolov8n-seg.yaml")
    model.load("yolov8n-seg.pt")
    return model


# Adiciona callback para detectar colapso durante o treino.
def attach_collapse_callback(model):
    def check_collapse(trainer):
        if trainer.epoch > 15:
            seg_loss = float(
                trainer.label_loss_items(trainer.tloss, prefix="train").get(
                    "train/seg_loss", 0
                )
            )
            map50 = float(trainer.metrics.get("metrics/mAP50(M)", 1))
            if seg_loss > 100 or map50 < 0.01:
                trainer.epoch = trainer.epochs

    model.callbacks["on_train_epoch_end"] = []
    model.add_callback("on_train_epoch_end", check_collapse)


# Executa treino e avaliação para cada combinação do grid.
def run_grid_search():
    for opt_name, act_name in itertools.product(optimizers_to_test, activations_to_test):
        run_name = f"yolo_seg_{opt_name}_{act_name}"
        current_lr0 = optimizer_hyperparams[opt_name]["lr0"]

        model = build_model_for_activation(act_name)
        attach_collapse_callback(model)

        BaseTrainer.force_custom_opt = opt_name
        model.train(
            data=dataset_yaml,
            epochs=150,
            patience=30,
            imgsz=640,
            batch=16,
            optimizer="auto",
            cos_lr=False,
            lr0=current_lr0,
            project=pasta_base,
            name=run_name,
            device="0",
            workers=4,
            exist_ok=True,
            amp=False,
            verbose=False,
        )

        metrics = model.val(
            data=dataset_yaml,
            split="test",
            project=pasta_base,
            name=f"{run_name}_teste_ETIS",
            exist_ok=True,
            verbose=False,
        )

        json_path = os.path.join(
            pasta_base, f"{run_name}_teste_ETIS", "etis_metrics.json"
        )
        with open(json_path, "w") as f:
            json.dump(extract_seg_metrics(metrics), f, indent=2)

        del model
        cleanup_memory()


run_grid_search()

In [ ]:
# YAMLs corrigidos para a divisão do dataset
TRAIN_YAML = "/workspace/projeto/Scripts/main_clean.yaml"  # ETIS = test
CVC_YAML = "/workspace/projeto/Scripts/cvc_eval.yaml"      # CVC = val
BASE_DIR = "/workspace/projeto/MBA_Variants_clean"

# LeakyReLU usado no treino
Conv.default_act = nn.LeakyReLU(negative_slope=0.1, inplace=True)

# Modelos a avaliar
models_to_eval = [
    "yolo8n-seg.pt",
    "yolo8s-seg.pt",
    "yolo8m-seg.pt",
    "yolo11n-seg.pt",
    "yolo11s-seg.pt",
    "yolo11m-seg.pt",
    "yolo26n-seg.pt",
    "yolo26s-seg.pt",
    "yolo26m-seg.pt",
]

results_list = []

for model_name in models_to_eval:
    run_name = f"yolo_variant_{model_name.replace('.pt', '')}_clean"
    best_pt = os.path.join(BASE_DIR, run_name, "weights", "best.pt")

    model = YOLO(best_pt)

    # Avaliação em ETIS
    metrics_etis = model.val(
        data=TRAIN_YAML,
        split="test",
        project=BASE_DIR,
        name=f"{run_name}_ETIS_eval",
        exist_ok=True,
        verbose=False,
    )

    # Avaliação em CVC
    metrics_cvc = model.val(
        data=CVC_YAML,
        split="val",
        project=BASE_DIR,
        name=f"{run_name}_CVC_eval",
        exist_ok=True,
        verbose=False,
    )

    entry = {
        "model": model_name,
        "mAP50_ETIS": round(float(metrics_etis.seg.map50), 4),
        "mAP50_CVC": round(float(metrics_cvc.seg.map50), 4),
        "recall_ETIS": round(float(metrics_etis.seg.mr), 4),
        "precision_ETIS": round(float(metrics_etis.seg.mp), 4),
    }
    results_list.append(entry)

    # Salva métricas individuais
    json_path = os.path.join(BASE_DIR, f"{run_name}_eval_metrics.json")
    with open(json_path, "w") as f:
        json.dump(entry, f, indent=2)


    # Limpa memória
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Tabela final
df = pd.DataFrame(results_list).sort_values("mAP50_ETIS", ascending=False)
df.to_csv(os.path.join(BASE_DIR, "variants_comparison_final.csv"), index=False)

In [ ]:
PRUNE_RATIO = 0.30  # 30% dos pesos zerados nas Conv do backbone
OUTPUT = "/workspace/projeto/MBA_Variants_clean/optimization_results"
MODELS = {
    "yolo11n-seg": "/workspace/projeto/MBA_Variants_clean/yolo_variant_yolo11n-seg_clean/weights/best.pt",
    "yolo26n-seg": "/workspace/projeto/MBA_Variants_clean/yolo_variant_yolo26n-seg_clean/weights/best.pt",
}


def calc_sparsity(model):
    """Calcula a esparsidade global do modelo (% de pesos zerados)"""
    total, zeros = 0, 0
    for p in model.parameters():
        total += p.numel()
        zeros += (p == 0).sum().item()
    return zeros / total


for name, pt_path in MODELS.items():
    model = YOLO(pt_path)
    pt_model = model.model

    params_before = sum(p.numel() for p in pt_model.parameters())
    sparsity_before = calc_sparsity(pt_model)

    # ── aplica poda L1 apenas nas Conv do backbone (layers 0–10) ──
    pruned_layers = []
    for layer_name, module in pt_model.named_modules():
        # poda apenas Conv2d do backbone — evita a cabeça Segment [23]
        if isinstance(module, nn.Conv2d) and not any(
            f"model.{i}." in layer_name or layer_name.endswith(f"model.{i}")
            for i in [23]  # protege a cabeça
        ):
            prune.l1_unstructured(module, name="weight", amount=PRUNE_RATIO)
            prune.remove(module, "weight")  # torna permanente
            pruned_layers.append(layer_name)

    params_after = sum(p.numel() for p in pt_model.parameters())
    sparsity_after = calc_sparsity(pt_model)


    # ── salva modelo completo (não só state_dict) ──────────────
    pruned_path = f"{OUTPUT}/{name}_pruned_l1_{int(PRUNE_RATIO*100)}pct.pt"
    ckpt = {"model": pt_model, "pruned_ratio": PRUNE_RATIO, "sparsity": sparsity_after}
    torch.save(ckpt, pruned_path)

for name in MODELS.keys():

    # Carrega o modelo podado e salva em formato compatível com Ultralytics
    checkpoint = torch.load(pruned_path, map_location="cpu", weights_only=False)
    pruned_obj = checkpoint["model"]

    ready_path = f"{OUTPUT}/{name}_pruned_l1_ready.pt"
    torch.save({"model": pruned_obj}, ready_path)

    model = YOLO(ready_path)
    model.train(
        data=TRAIN_YAML,
        epochs=50,
        imgsz=640,
        batch=16,
        optimizer="AdaBelief",
        lr0=0.0005,  # LR conservador — modelo já convergiu
        patience=20,
        project=OUTPUT,
        name=f"finetune_l1_{name}",
        exist_ok=True,
        device=0,
    )

In [ ]:

PAIRS = [
    {
        "name": "yolo11n-seg",
        "student_pt": f"{OUTPUT}/finetune_l1_yolo11n-seg/weights/best.pt",
        "teacher_pt": "/workspace/projeto/MBA_Variants_clean/yolo_variant_yolo11m-seg_clean/weights/best.pt",
        "run_name": "distill_yolo11m_to_yolo11n",
    },
    {
        "name": "yolo26n-seg",
        "student_pt": f"{OUTPUT}/finetune_l1_yolo26n-seg/weights/best.pt",
        "teacher_pt": "/workspace/projeto/MBA_Variants_clean/yolo_variant_yolo26m-seg_clean/weights/best.pt",
        "run_name": "distill_yolo26m_to_yolo26n",
    },
]

ALPHA = 0.5
TEMP = 4.0

for pair in PAIRS:
    print(f"\n🎓 Destilação: {pair['name']}")

    teacher = YOLO(pair["teacher_pt"]).model.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    teacher = teacher.to("cuda:0")

    student_model = YOLO(pair["student_pt"])

    def make_distill_callback(teacher, alpha, temp):
        def on_train_batch_end(trainer):
            # ✅ batch disponível aqui — contém [imgs, labels, ...]
            if not hasattr(trainer, "batch") or trainer.batch is None:
                return

            imgs = trainer.batch[0].to("cuda:0").float() / 255.0

            with torch.no_grad():
                t_out = teacher(imgs)

            s_out = trainer.model(imgs)

            t_feat = t_out[0] if isinstance(t_out, (list, tuple)) else t_out
            s_feat = s_out[0] if isinstance(s_out, (list, tuple)) else s_out

            # alinha dimensões se necessário
            if t_feat.shape != s_feat.shape:
                if t_feat.dim() == 4 and s_feat.dim() == 4:
                    t_feat = F.adaptive_avg_pool2d(
                        t_feat, (s_feat.shape[-2], s_feat.shape[-1])
                    )
                else:
                    # dimensões incompatíveis — pula essa iteração
                    return

            t_soft = F.log_softmax(t_feat.flatten(1) / temp, dim=-1)
            s_soft = F.log_softmax(s_feat.flatten(1) / temp, dim=-1)
            loss_kd = F.kl_div(s_soft, t_soft.exp(), reduction="batchmean") * (temp**2)

            trainer.loss = trainer.loss + alpha * loss_kd

        return on_train_batch_end

    # ✅ remove o callback on_train_batch_start — não é mais necessário
    student_model.add_callback(
        "on_train_batch_end", make_distill_callback(teacher, ALPHA, TEMP)
    )

    student_model.train(
        data=TRAIN_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        optimizer="AdaBelief",
        lr0=0.0005,
        patience=30,
        project=OUTPUT,
        name=pair["run_name"],
        exist_ok=True,
        device=0,
    )

    del teacher
    torch.cuda.empty_cache()

In [ ]:
BASE_PT1 = f"{OUTPUT}/distill_yolo11m_to_yolo11n/weights/best.pt"

model = YOLO(BASE_PT1)
model.train(
    data=TRAIN_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=1e-4,
    patience=10,
    project=f"{OUTPUT}/extra_finetune_yolo11n",
    name="run",
    exist_ok=False,
)

BASE_PT2 = f"{OUTPUT}/distill_yolo26m_to_yolo26n/weights/best.pt"

model = YOLO(BASE_PT2)
model.train(
    data=TRAIN_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=5e-5,
    lrf=0.01,
    patience=10,
    project=f"{OUTPUT}/extra_finetune_yolo26n",
    name="run",
    exist_ok=False,
)

In [ ]:
MODELS = {
    "yolo11n-seg": "MBA_Variants_clean/optimization_results/extra_finetune_yolo11n/run/weights/best.pt",
    "yolo26n-seg": "MBA_Variants_clean/optimization_results/extra_finetune_yolo26n/run/weights/best.pt",
}


for name, model_path in MODELS.items():
    base = model_path.replace(".pt", "")
    paths = {
        "onnx_fp32":    base + "_fp32.onnx",
        "onnx_fp16":    base + "_fp16.onnx",
        "engine_fp16":  base + "_fp16.engine",
    }
    row = {}

    # ── PyTorch GPU ───────────────────────────────────────────
    model = YOLO(model_path)
    model.to("cuda")
    mean, std = bench(lambda: model(dummy.cuda(), verbose=False))
    print(f"PyTorch GPU:       {mean:.2f} ms ± {std:.2f}")
    row["pytorch_gpu"] = mean

    # ── ONNX FP32 CPU ─────────────────────────────────────────
    model.export(format="onnx", half=False, imgsz=640)
    os.rename(base + ".onnx", paths["onnx_fp32"])
    sess32 = ort.InferenceSession(paths["onnx_fp32"], providers=["CPUExecutionProvider"])
    inp32 = {sess32.get_inputs()[0].name: dummy.numpy()}
    mean, std = bench(lambda: sess32.run(None, inp32))
    print(f"ONNX FP32 CPU:     {mean:.2f} ms ± {std:.2f}")
    row["onnx_fp32_cpu"] = mean

    # ── ONNX FP16 CPU ─────────────────────────────────────────
    model.export(format="onnx", half=True, imgsz=640)
    os.rename(base + ".onnx", paths["onnx_fp16"])
    sess16 = ort.InferenceSession(paths["onnx_fp16"], providers=["CPUExecutionProvider"])
    inp16 = {sess16.get_inputs()[0].name: dummy.numpy()}  # ONNX Runtime aceita fp32 mesmo com modelo fp16
    mean, std = bench(lambda: sess16.run(None, inp16))
    print(f"ONNX FP16 CPU:     {mean:.2f} ms ± {std:.2f}")
    row["onnx_fp16_cpu"] = mean

    # ── TensorRT FP16 GPU ─────────────────────────────────────
    model.export(format="engine", half=True, imgsz=640)
    os.rename(base + ".engine", paths["engine_fp16"])
    model_trt = YOLO(paths["engine_fp16"])
    mean, std = bench(lambda: model_trt(dummy.cuda(), verbose=False))
    print(f"TensorRT FP16 GPU: {mean:.2f} ms ± {std:.2f}")
    row["trt_fp16_gpu"] = mean

    results[name] = row